# X-OPT - STRUCTURAL INTERPRETABILITY BENCHMARK

This notebook benchmarks the post-hoc structural interpretability pipeline over the OR-Library p-median instances. For each instance, the notebook performs the following steps:

1. run the metaheuristic;
2. build the long-term memory (LTM);
3. keep the top 10% best LTM solutions;
4. build the weighted co-occurrence graph;
5. extract communities, max-p-cut, k-core, and densest subgraph;
6. compute the requested interpretability metrics;
7. display and save the final CSV table.

### SETUP

Importing the libraries:

In [1]:
from __future__ import annotations

import sys

import numpy    as np
import pandas   as pd
import networkx as nx

from pathlib             import Path
from tqdm.auto           import tqdm
from networkx.algorithms import community
from time                import perf_counter


pd.set_option("display.max_columns" , None)
pd.set_option("display.max_colwidth", 120 )

In [2]:
from lib.paths     import find_project_root      , \
                          instance_sort_key      , \
                          canonical_instance_name

from lib.instances import list_orlibrary_instances, \
                          read_instance_metadata  , \
                          load_best_known_costs

from lib.graph     import build_top_ltm            , \
                          build_cooccurrence_matrix, \
                          build_weighted_graph     , \
                          build_unweighted_graph   , \
                          total_edge_weight        , \
                          total_edge_count

from lib.maxcut    import max_p_cut_local_search  , \
                          best_facility_separation

from lib.explain   import densest_subgraph_greedy
from lib.metrics   import compute_gap_percent

Adding a new path to the Python interpreter:

In [3]:
PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

PROJECT_ROOT = /home/rei-luisinho/xopt


In [4]:
import pymedian

### EXPERIMENT CONFIGURATION

In [5]:
INSTANCES_DIR = PROJECT_ROOT  / "instances"
PMEDOPT_PATH  = INSTANCES_DIR / "pmedopt.txt"

OUTPUT_DIR   = PROJECT_ROOT / "notebooks" / "experiments_sbpo" / "artifacts"
RESULTS_CSV  = OUTPUT_DIR   / "structural_interpretability_benchmark.csv"
FAILURES_CSV = OUTPUT_DIR   / "structural_interpretability_benchmark_failures.csv"

DEFAULT_INSTANCE_NAMES = list_orlibrary_instances(INSTANCES_DIR)
INSTANCE_NAMES         = DEFAULT_INSTANCE_NAMES


RESTARTS = 8
MAX_ITER = 25
FACTOR   = 1
DETAILS_FORMAT = "binary"

TOP_FRACTION = 0.1

GLOBAL_SEED        = 42
MAX_P_CUT_RESTARTS = 10
MAX_P_CUT_MAX_ITER = 1000


SAVE_RESULTS_CSV  = True
SAVE_FAILURES_CSV = True

BEST_KNOWN_COSTS_DF = load_best_known_costs(PMEDOPT_PATH)
BEST_KNOWN_COSTS    = BEST_KNOWN_COSTS_DF.set_index("instance_id")["best_known_cost"].to_dict()

Verifying the hyperparameters:

In [6]:
print(f"Project root     : {PROJECT_ROOT }")
print(f"Instances folder : {INSTANCES_DIR}")
print(f"Results CSV      : {RESULTS_CSV  }")

Project root     : /home/rei-luisinho/xopt
Instances folder : /home/rei-luisinho/xopt/instances
Results CSV      : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/structural_interpretability_benchmark.csv


In [7]:
print(f"Number of instances            : {len(INSTANCE_NAMES)}")
print(f"Top fraction kept from the LTM : {TOP_FRACTION   :.0%}")
print(f"Metaheuristic parameters       : restarts={RESTARTS       :2d}, max_iter={MAX_ITER       :4d}, factor={FACTOR     }")
print(f"Max-p-Cut parameters           : restarts={MAX_P_CUT_RESTARTS}, max_iter={MAX_P_CUT_MAX_ITER}, seed  ={GLOBAL_SEED}")

Number of instances            : 40
Top fraction kept from the LTM : 10%
Metaheuristic parameters       : restarts= 8, max_iter=  25, factor=1
Max-p-Cut parameters           : restarts=10, max_iter=1000, seed  =42


In [8]:
selected_instances_df = pd.DataFrame(
    [
        {
            "instance"        :      canonical_instance_name(instance_name)      ,
            "instance_id"     : Path(canonical_instance_name(instance_name)).stem,

            "best_known_cost" : BEST_KNOWN_COSTS.get(
                Path(canonical_instance_name(instance_name)).stem, np.nan,
            ),

            **read_instance_metadata(
                INSTANCES_DIR / canonical_instance_name(instance_name))
            ,
        }
        for instance_name in INSTANCE_NAMES
    ]
)

selected_instances_df["instance_order"] = selected_instances_df["instance"].map(
    lambda value: instance_sort_key(value)[0]
)

selected_instances_df = (
    selected_instances_df.sort_values(["instance_order", "instance"])
                         .drop       (columns="instance_order"      )
                         .reset_index(drop   =True)
)

display(selected_instances_df.head(10))

,instance,instance_id,best_known_cost,n,p
0,pmed1.txt,pmed1,5819.0,100,5
1,pmed2.txt,pmed2,4093.0,100,10
2,pmed3.txt,pmed3,4250.0,100,10
3,pmed4.txt,pmed4,3034.0,100,20
4,pmed5.txt,pmed5,1355.0,100,33
5,pmed6.txt,pmed6,7824.0,200,5
6,pmed7.txt,pmed7,5631.0,200,10
7,pmed8.txt,pmed8,4445.0,200,20
8,pmed9.txt,pmed9,2734.0,200,40
9,pmed10.txt,pmed10,1255.0,200,67


### METRIC DEFINITIONS

Let:

- $B$ be the set of facilities in the best solution returned by the metaheuristic;
- $G_{top} = (V, E_{top})$ be the weighted co-occurrence graph built from the top 5% LTM solutions;
- $A$ be its weighted adjacency matrix;
- $C_1, \ldots, C_k$ be the communities found in $G_{top}$;
- $D \subseteq V$ be the node set of the densest subgraph;
- $K_{\max}$ be the highest k-core.

The benchmark uses the following definitions.

#### COMMUNITIES

- **Weighted modularity**

It's the most natural metric if you're using community detection in a weighted graph; the higher the $Q$ value, the more coherent the partition.

$$
Q_w = \frac{1}{2m}\sum_{i,j}\left(A_{ij} - \frac{k_i^w k_j^w}{2m}\right)\mathbf{1}[c_i = c_j],
$$

where $k_i^w = \sum_j A_{ij}$ and $2m = \sum_{i,j} A_{ij}$.

- **Coverage**

Fraction of the total weight of the edges that remains within the communities.

$$
\mathrm{Cov} = \frac{\sum_{g=1}^{k}\sum_{i<j,\ i,j\in C_g} A_{ij}}{\sum_{i<j} A_{ij}}
$$

- **Best-facility concentration**

If $B$ is the set of the $p$ facilities in the best solution, then:

$$
\mathrm{BFC} = \sum_{g=1}^{k}\left(\frac{|B \cap C_g|}{|B|}\right)^2
$$

- close to $1$: the best facilities are concentrated in few communities;
- lower values: the best facilities are spread across many communities.

- **Best-facility purity**

This is the percentage of communities with size at least $2$ that contain at least one best facility:

$$
\mathrm{BFP} =
\frac{\left|\left\{g : |C_g| \ge 2,\ B \cap C_g \neq \emptyset\right\}\right|}
     {\left|\left\{g : |C_g| \ge 2\right\}\right|}
$$

In [9]:
def community_internal_weight(
    adjacency : np.ndarray               ,
    nodes     : set[int] | frozenset[int],
) -> float:
    node_array = np.array(sorted(nodes), dtype=int)

    if node_array.size <= 1:
        return 0.0

    subgraph = adjacency[np.ix_(node_array, node_array)]

    return float(np.triu(subgraph, k=1).sum())


def weighted_coverage(
    adjacency         : np.ndarray                     ,
    communities_found : list[set[int] | frozenset[int]],
) -> float:
    total_weight = total_edge_weight(adjacency)

    if total_weight <= 0.0:
        return 0.0

    internal_weight = sum(
        community_internal_weight(
            adjacency, community_nodes
        )
        for community_nodes in communities_found
    )

    return internal_weight / total_weight


def best_facility_concentration(
    communities_found : list[set[int] | frozenset[int]],
    best_set          : set [int]                      ,
) -> float:
    if not best_set:
        return 0.0

    best_size = len(best_set)

    return float(
        sum(
            (len(set(community_nodes).intersection(best_set)) / best_size) ** 2
            for community_nodes in communities_found
        )
    )


def best_facility_purity(
    communities_found : list[set[int] | frozenset[int]],
    best_set          : set [int]                      ,
) -> float:
    eligible_communities = [
        set(community_nodes)
        for community_nodes in communities_found
        if  len(community_nodes) >= 2
    ]

    if not eligible_communities:
        return 0.0

    communities_with_best = sum(
        1
        for community_nodes in eligible_communities
        if  community_nodes.intersection(best_set)
    )

    return communities_with_best / len(eligible_communities)


def community_metrics(
    graph     : nx.Graph  ,
    adjacency : np.ndarray,
    best_set  : set[int]  ,
) -> dict[str, float | int]:
    if graph.number_of_edges() == 0:
        communities_found   = [
            frozenset([node])
            for node in graph.nodes
        ]
        weighted_modularity = 0.0
    else:
        communities_found   = list (
            community.greedy_modularity_communities(graph, weight="weight")
        )
        weighted_modularity = float(
            nx.community.modularity(graph, communities_found, weight="weight")
        )

    return {
        "communities_count"               : len(communities_found),
        "communities_weighted_modularity" : weighted_modularity   ,
        "communities_coverage"            : weighted_coverage(adjacency, communities_found),

        "communities_best_facility_concentration" : best_facility_concentration(communities_found, best_set),
        "communities_best_facility_purity"        : best_facility_purity       (communities_found, best_set),
    }

#### MAX-p-CUT

- **Fraction cut**

$$
\mathrm{FC} = \frac{W_{\mathrm{cut}}}{W_{\mathrm{cut}} + W_{\mathrm{int}}}
$$

- **Best-facility separation**

$$
\mathrm{BFS} =
\frac{1}{\binom{|B|}{2}}
\sum_{\{i,j\}\subseteq B}\mathbf{1}[\ell_i \ne \ell_j]
$$

where $\ell_i$ is the Max-p-Cut label assigned to facility $i$.

#### K-CORE

- **Core mass**

$$
\mathrm{CM} = \frac{|K_{\max}|}{|V|}
$$

- **Best-core recall**

$$
\mathrm{BCR} = \frac{|B \cap K_{\max}|}{|B|}
$$

- **Best-core precision**

$$
\mathrm{BCP} = \frac{|B \cap K_{\max}|}{|K_{\max}|}
$$

In [10]:
def k_core_metrics(
    graph    : nx.Graph,
    n        : int     ,
    best_set : set[int],
) -> dict[str, float | int]:
    core_numbers   = nx.core_number(graph)

    max_core_level = max(
        core_numbers.values()
    ) if core_numbers else 0

    highest_core_nodes = {
        node
        for node, core_level in core_numbers.items()
        if  core_level == max_core_level
    }

    overlap = highest_core_nodes.intersection(best_set)

    return {
        "k_core_max_level" : int(max_core_level    ),
        "k_core_core_size" : len(highest_core_nodes),
        "k_core_core_mass" : len(highest_core_nodes) / max(1, n),

        "k_core_best_core_recall"    : len(overlap) / max(1, len(best_set          )),
        "k_core_best_core_precision" : len(overlap) / max(1, len(highest_core_nodes)),
    }

#### DENSEST SUBGRAPH

- **Average internal weighted degree**

$$
\mathrm{AIWD} = \frac{1}{|D|}\sum_{i\in D}\sum_{j\in D} A_{ij}
$$

- **Best overlap**

$$
\mathrm{BO} = \frac{|D \cap B|}{|B|}
$$

- **Original-edge percentage**

$$
\mathrm{OEP} = \frac{|E(D)|}{|E_{top}|}
$$

where $E(D)$ is the set of positive-weight edges induced by $D$ inside the original co-occurrence graph.

In [11]:
def induced_edge_count(
    adjacency : np.ndarray,
    nodes     : set[int]  ,
) -> int:
    if len(nodes) <= 1:
        return 0

    node_list = sorted(nodes)
    subgraph  = adjacency[np.ix_(node_list, node_list)]

    return int(
        np.count_nonzero(np.triu(subgraph, k=1) > 0)
    )


def average_internal_weighted_degree(
    adjacency : np.ndarray,
    nodes     : set[int]  ,
) -> float:
    if not nodes:
        return 0.0

    node_list        = sorted(nodes)
    subgraph         = adjacency[np.ix_(node_list, node_list)]
    weighted_degrees = subgraph.sum(axis=1)

    return float(np.mean(weighted_degrees))


def jaccard_overlap(
    set_a: set[int],
    set_b: set[int],
) -> float:
    union = set_a.union(set_b)

    if not union:
        return 1.0

    return len(set_a.intersection(set_b)) / len(set_a)


def densest_subgraph_edge_percentage(
    adjacency     : np.ndarray,
    densest_nodes : set[int]  ,
) -> float:
    total_edges = total_edge_count(adjacency)

    if total_edges <= 0:
        return 0.0

    return induced_edge_count(adjacency, densest_nodes) / total_edges


def densest_subgraph_metrics(
    adjacency : np.ndarray,
    best_set  : set[int]  ,
) -> dict[str, float | int]:
    densest_nodes, _ = densest_subgraph_greedy(
        adjacency, min_size=3
    )

    return {
        "densest_subgraph_size" : len(densest_nodes),

        "densest_subgraph_average_internal_weighted_degree" : average_internal_weighted_degree(adjacency, densest_nodes),
        "densest_subgraph_edge_percentage"                  : densest_subgraph_edge_percentage(adjacency, densest_nodes),
        "densest_subgraph_best_overlap"                     : jaccard_overlap                 (best_set , densest_nodes),
    }

#### IMPLEMENTATION

In [12]:
def run_single_instance_analysis(
    instance_name: str,
    *,
    restarts     : int  ,
    max_iter     : int  ,
    factor       : int  ,
    top_fraction : float,

    details_format     : str = "binary",
    max_p_cut_restarts : int = 30  ,
    max_p_cut_max_iter : int = 2000,
    global_seed        : int = 42  ,
) -> dict[str, object]:
    instance_name = canonical_instance_name(instance_name)
    instance_path = INSTANCES_DIR / instance_name
    instance_id   = Path(instance_name).stem

    row = {
        "instance"    : instance_name,
        "instance_id" : instance_id  ,
        "n"           : np.nan,
        "p"           : np.nan,

        "best_known_cost"                      : BEST_KNOWN_COSTS.get(instance_id, np.nan),
        "best_cost"                            : np.nan,
        "gap_percent"                          : np.nan,
        "memory_size"                          : np.nan,
        "top_solution_count"                   : np.nan,
        "top_cost_cutoff"                      : np.nan,
        "cooccurrence_edges"                   : np.nan,
        "cooccurrence_total_weight"            : np.nan,
        "metaheuristic_runtime_seconds"        : np.nan,
        "structure_extraction_runtime_seconds" : np.nan,
        "total_pipeline_runtime_seconds"       : np.nan,

        "communities_count"                       : np.nan,
        "communities_weighted_modularity"         : np.nan,
        "communities_coverage"                    : np.nan,
        "communities_best_facility_concentration" : np.nan,
        "communities_best_facility_purity"        : np.nan,

        "max_p_cut_fraction_cut"             : np.nan,
        "max_p_cut_best_facility_separation" : np.nan,

        "k_core_max_level"           : np.nan,
        "k_core_core_size"           : np.nan,
        "k_core_core_mass"           : np.nan,
        "k_core_best_core_recall"    : np.nan,
        "k_core_best_core_precision" : np.nan,

        "densest_subgraph_size"                             : np.nan,
        "densest_subgraph_average_internal_weighted_degree" : np.nan,
        "densest_subgraph_edge_percentage"                  : np.nan,
        "densest_subgraph_best_overlap"                     : np.nan,

        "status"        : "error",
        "error_message" : None   ,
    }

    if not instance_path.exists():
        row["error_message"] = f"Instance not found: {instance_path}"

        return row

    metadata = read_instance_metadata(instance_path)
    row["n"] = metadata["n"]
    row["p"] = metadata["p"]

    overall_started_at = perf_counter()

    try:
        meta_started_at = perf_counter()

        summary, details = pymedian.solve_pmedian(
            instance_path,
            restarts      =restarts      ,
            max_iter      =max_iter      ,
            factor        =factor        ,
            details_format=details_format,
        )

        metaheuristic_runtime_seconds = perf_counter() - meta_started_at

        long_term_memory = details["long_term_memory"]
        if not long_term_memory:
            raise ValueError("long_term_memory is empty.")

        analysis_ltm, matrix, costs = build_top_ltm            (long_term_memory, top_fraction)
        adjacency                   = build_cooccurrence_matrix(matrix)

        weighted_graph   = build_weighted_graph  (adjacency)
        unweighted_graph = build_unweighted_graph(adjacency)

        best_facilities = tuple(
            sorted(
                int(value) - 1
                for value in summary["tspmed_facilities"]
            )
        )
        best_set        = set(best_facilities)

        structure_started_at = perf_counter()

        community_stats = community_metrics(weighted_graph, adjacency, best_set)

        labels_maxcut, cut_weight, internal_weight = max_p_cut_local_search(
            adjacency        ,
            int(summary["p"]),
            n_restarts=max_p_cut_restarts,
            max_iter  =max_p_cut_max_iter,
            seed      =global_seed       ,
        )

        max_p_cut_stats = {
            "max_p_cut_fraction_cut"             : cut_weight / max(1e-12, cut_weight + internal_weight),
            "max_p_cut_best_facility_separation" : best_facility_separation(labels_maxcut, best_set    ),
        }

        k_core_stats = k_core_metrics(
            unweighted_graph ,
            int(summary["n"]),
            best_set         ,
        )

        densest_stats = densest_subgraph_metrics(adjacency, best_set)

        structure_extraction_runtime_seconds = perf_counter() - structure_started_at
        total_pipeline_runtime_seconds       = perf_counter() - overall_started_at

        row.update(
            {
                "n"           : int  (summary["n"          ]),
                "p"           : int  (summary["p"          ]),
                "best_cost"   : float(summary["tspmed_cost"]),
                "gap_percent" : compute_gap_percent(
                    summary["tspmed_cost"    ],
                    row    ["best_known_cost"],
                ),

                "memory_size"               : len  (long_term_memory                ),
                "top_solution_count"        : len  (analysis_ltm                    ),
                "top_cost_cutoff"           : float(costs.max()                     ),
                "cooccurrence_edges"        : int  (weighted_graph.number_of_edges()),
                "cooccurrence_total_weight" : total_edge_weight(adjacency           ),

                "metaheuristic_runtime_seconds"        : metaheuristic_runtime_seconds       ,
                "structure_extraction_runtime_seconds" : structure_extraction_runtime_seconds,
                "total_pipeline_runtime_seconds"       : total_pipeline_runtime_seconds      ,

                **community_stats,
                **max_p_cut_stats,
                **k_core_stats   ,
                **densest_stats  ,

                "status"        : "ok",
                "error_message" : None,
            }
        )
    except Exception as exc:
        row["total_pipeline_runtime_seconds"] = perf_counter() - overall_started_at
        row["error_message"                 ] = f"{type(exc).__name__}: {exc}"

    return row


def run_benchmark(
    instance_names: list[str],
    *,
    restarts     : int  ,
    max_iter     : int  ,
    factor       : int  ,
    top_fraction : float,

    details_format     : str = "binary",
    max_p_cut_restarts : int = 30  ,
    max_p_cut_max_iter : int = 2000,
    global_seed        : int = 42  ,
) -> pd.DataFrame:
    if not instance_names:
        raise ValueError(
            "The benchmark requires at least one instance."
        )

    rows = []

    for instance_name in tqdm(
        instance_names,
        total        =len(instance_names)   ,
        desc         ="Structural benchmark",
        dynamic_ncols=True                  ,
    ):
        rows.append(
            run_single_instance_analysis(
                instance_name,
                restarts          =restarts          ,
                max_iter          =max_iter          ,
                factor            =factor            ,
                top_fraction      =top_fraction      ,
                details_format    =details_format    ,
                max_p_cut_restarts=max_p_cut_restarts,
                max_p_cut_max_iter=max_p_cut_max_iter,
                global_seed       =global_seed       ,
            )
        )

    results_df = pd.DataFrame(rows)

    results_df["interpretability_overhead_ratio"] = np.where(
        results_df["metaheuristic_runtime_seconds"       ] > 0,
        results_df["structure_extraction_runtime_seconds"] / 
        results_df["metaheuristic_runtime_seconds"       ] ,
        np.nan,
    )

    results_df["instance_order"] = results_df["instance"].map(
        lambda value: instance_sort_key(value)[0]
    )

    return (
        results_df.sort_values(["instance_order", "instance"])
                  .drop       (columns="instance_order"      )
                  .reset_index(drop   =True)
    )

### APPLY

In [13]:
results_df = run_benchmark(
    INSTANCE_NAMES,
    restarts          =RESTARTS          ,
    max_iter          =MAX_ITER          ,
    factor            =FACTOR            ,
    top_fraction      =TOP_FRACTION      ,
    details_format    =DETAILS_FORMAT    ,
    max_p_cut_restarts=MAX_P_CUT_RESTARTS,
    max_p_cut_max_iter=MAX_P_CUT_MAX_ITER,
    global_seed       =GLOBAL_SEED       ,
)

Structural benchmark:   0%|          | 0/40 [00:00<?, ?it/s]

In [14]:
final_column_order = [
    "instance"   ,
    "instance_id",
    "n",
    "p",
    "best_known_cost",
    "best_cost"      ,
    "gap_percent"    ,
    "memory_size"    ,

    "top_solution_count"       ,
    "top_cost_cutoff"          ,
    "cooccurrence_edges"       ,
    "cooccurrence_total_weight",

    "metaheuristic_runtime_seconds"       ,
    "structure_extraction_runtime_seconds",
    "interpretability_overhead_ratio"     ,
    "total_pipeline_runtime_seconds"      ,

    "communities_count"                      ,
    "communities_weighted_modularity"        ,
    "communities_coverage"                   ,
    "communities_best_facility_concentration",
    "communities_best_facility_purity"       ,

    "max_p_cut_fraction_cut"            ,
    "max_p_cut_best_facility_separation",

    "k_core_max_level"          ,
    "k_core_core_size"          ,
    "k_core_core_mass"          ,
    "k_core_best_core_recall"   ,
    "k_core_best_core_precision",

    "densest_subgraph_size"                            ,
    "densest_subgraph_average_internal_weighted_degree",
    "densest_subgraph_edge_percentage"                 ,
    "densest_subgraph_best_overlap"                    ,

    "status"       ,
    "error_message",
]

final_results_df = results_df[final_column_order].copy()

display(final_results_df)

,instance,instance_id,n,p,best_known_cost,best_cost,gap_percent,memory_size,top_solution_count,top_cost_cutoff,cooccurrence_edges,cooccurrence_total_weight,metaheuristic_runtime_seconds,structure_extraction_runtime_seconds,interpretability_overhead_ratio,total_pipeline_runtime_seconds,communities_count,communities_weighted_modularity,communities_coverage,communities_best_facility_concentration,communities_best_facility_purity,max_p_cut_fraction_cut,max_p_cut_best_facility_separation,k_core_max_level,k_core_core_size,k_core_core_mass,k_core_best_core_recall,k_core_best_core_precision,densest_subgraph_size,densest_subgraph_average_internal_weighted_degree,densest_subgraph_edge_percentage,densest_subgraph_best_overlap,status,error_message
0,pmed1.txt,pmed1,100,5,5819.0,5819.0,0.000000,129,13,5856.0,40,130.0,0.033368,0.024378,0.730588,0.059410,91,7.573964e-03,0.815385,1.000000,0.500000,1.000000,1.000000,6,8,0.080000,1.000000,0.625000,7,26.857143,0.475000,1.000000,ok,None
1,pmed2.txt,pmed2,100,10,4093.0,4093.0,0.000000,107,11,4101.0,95,495.0,0.058286,0.015664,0.268745,0.075274,86,0.000000e+00,1.000000,1.000000,1.000000,1.000000,1.000000,11,13,0.130000,1.000000,0.769231,11,73.818182,0.568421,1.000000,ok,None
2,pmed3.txt,pmed3,100,10,4250.0,4250.0,0.000000,102,11,4255.0,119,495.0,0.051485,0.019520,0.379135,0.072143,84,7.254362e-03,0.771717,0.820000,1.000000,1.000000,1.000000,10,13,0.130000,1.000000,0.769231,9,72.666667,0.302521,0.900000,ok,None
3,pmed4.txt,pmed4,100,20,3034.0,3034.0,0.000000,101,11,3034.0,271,2090.0,0.088563,0.020035,0.226219,0.109568,77,0.000000e+00,1.000000,0.905000,1.000000,1.000000,0.994737,21,24,0.240000,0.950000,0.791667,22,177.909091,0.845018,0.900000,ok,None
4,pmed5.txt,pmed5,100,33,1355.0,1355.0,0.000000,87,9,1355.0,661,4752.0,0.105750,0.025480,0.240950,0.132645,64,0.000000e+00,1.000000,0.829201,1.000000,1.000000,0.996212,34,37,0.370000,0.909091,0.810811,33,259.454545,0.798790,0.818182,ok,None
5,pmed6.txt,pmed6,200,5,7824.0,7824.0,0.000000,246,25,7874.0,61,250.0,0.187824,0.035008,0.186387,0.224890,186,0.000000e+00,1.000000,1.000000,1.000000,1.000000,1.000000,7,9,0.045000,1.000000,0.555556,7,46.571429,0.311475,1.000000,ok,None
6,pmed7.txt,pmed7,200,10,5631.0,5631.0,0.000000,112,12,5643.0,121,540.0,0.425933,0.081986,0.192485,0.511957,184,1.356481e-02,0.820370,1.000000,0.500000,1.000000,1.000000,10,16,0.080000,1.000000,0.625000,11,77.636364,0.446281,1.000000,ok,None
7,pmed8.txt,pmed8,200,20,4445.0,4445.0,0.000000,89,9,4447.0,269,1710.0,0.928213,0.058910,0.063466,0.989602,177,0.000000e+00,1.000000,1.000000,1.000000,1.000000,1.000000,21,23,0.115000,1.000000,0.869565,23,147.043478,0.929368,1.000000,ok,None
8,pmed9.txt,pmed9,200,40,2734.0,2734.0,0.000000,146,15,2734.0,1256,11700.0,0.936782,0.068801,0.073444,1.009943,150,6.968803e-03,0.908974,0.632500,1.000000,1.000000,0.996154,42,44,0.220000,0.750000,0.681818,40,512.850000,0.621019,0.725000,ok,None
9,pmed10.txt,pmed10,200,67,1255.0,1255.0,0.000000,173,18,1255.0,3163,39798.0,1.612212,0.105161,0.065228,1.768963,121,2.825213e-05,0.993417,0.913121,1.000000,1.000000,0.999095,70,75,0.375000,0.910448,0.813333,67,1089.223881,0.699020,0.820896,ok,None


In [15]:
success_df = final_results_df.loc[final_results_df["status"] == "ok"].copy()

failure_df = final_results_df.loc[
    final_results_df["status"] != "ok",
    [
        "instance"     ,
        "instance_id"  ,
        "status"       ,
        "error_message",
    ],
].reset_index(drop=True)


print(f"Successful instances : {len(success_df)}")
print(f"Failed instances     : {len(failure_df)}")

print()
if failure_df.empty:
    print  ("No execution failures were recorded.")
else:
    display(failure_df)

Successful instances : 40
Failed instances     : 0

No execution failures were recorded.


Saving the results:

In [16]:
if SAVE_RESULTS_CSV:
    OUTPUT_DIR      .mkdir (parents=True, exist_ok=True )
    final_results_df.to_csv(RESULTS_CSV , index   =False)

    print(f"Results saved to  : {RESULTS_CSV}")

if SAVE_FAILURES_CSV and not failure_df.empty:
    OUTPUT_DIR.mkdir (parents=True, exist_ok=True )
    failure_df.to_csv(FAILURES_CSV, index   =False)

    print(f"Failures saved to : {FAILURES_CSV}")

Results saved to  : /home/rei-luisinho/xopt/notebooks/experiments_sbpo/artifacts/structural_interpretability_benchmark.csv
